# Chapter 8: Model Evaluation and Tradeoffs

**Foundations of Data Science**

## In this lesson

1. The confusion matrix, and the four numbers a clinician asks for
2. Moving the cutoff: sensitivity against specificity, and who pays
3. ROC and precision-recall curves: the whole tradeoff on one page
4. Comparing two models honestly (DeLong)
5. Calibration: does 23% mean 23%?
6. Error bars on everything: the bootstrap again
7. The screening rule: turning a model into a policy the clinic can run

## The one idea

**The right metric follows from the cost of each kind of mistake.** A missed diabetic (false negative) costs years of untreated disease. An unnecessary test (false positive) costs a lab visit. Those are not equal, so a metric that treats them as equal (accuracy) is answering a question nobody asked. Every choice today, the cutoff, the curve, the summary number, starts from asking what a wrong answer costs.

Final form of the clinic question: **at what predicted probability do we order the test, how many people does that flag, and how many cases does it catch?**

## Setup

<!-- nb-links -->
[Download this notebook](week_08_evaluation.ipynb?download=1){download="week_08_evaluation.ipynb"} &nbsp;|&nbsp; [![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lshpaner/fds_course_students/blob/main/notebooks/week_08/week_08_evaluation.ipynb)


In [ ]:
# --- Setup: works in Colab and locally ---------------------------------------
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO = "fds_course"

if IN_COLAB:
    # Clone the course repo and move into it (skip if already there)
    if not Path(REPO).exists():
        !git clone -q https://github.com/lshpaner/{REPO}.git
    os.chdir(REPO)
    !pip install -q -r requirements.txt
else:
    # Local: the notebook lives in notebooks/week_XX/, so go up two levels to the repo root
    if Path.cwd().name.startswith("week_"):
        os.chdir(Path.cwd().parents[1])

# Make core/ importable
sys.path.insert(0, os.getcwd())

# Fetch this chapter's raw data only if it isn't already on disk, then build the processed table
for component in ["nhanes_demo", "nhanes_bmx", "nhanes_bpx", "nhanes_diq"]:
    if not Path(f"data/raw/{component}.parquet").exists():
        !python data_gen.py fetch --name {component}
if not Path("data/processed/nhanes.parquet").exists():
    !python preprocessing.py

print("Working directory:", os.getcwd())

## 1. Refit last chapter's two models

Same split, same preprocessing, same chosen hyperparameters (grids of one value so there is nothing to search). Logistic regression and XGBoost; the forest added nothing last chapter.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from model_tuner import Model

nhanes = pd.read_parquet("data/processed/nhanes.parquet")
adults = nhanes[nhanes["age"] >= 20].copy()
adults["sex"] = adults["sex"].map({1: "male", 2: "female"})
adults["race_eth"] = adults["race_eth"].map({1: "Mexican American", 2: "Other Hispanic", 3: "White", 4: "Black", 6: "Asian", 7: "Other/multi"})
num = ["age", "bmi", "waist_cm", "sbp", "dbp", "income_ratio"]
cat = ["sex", "race_eth"]
X, y = adults[num + cat], adults["diabetes"]

preprocess = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), num),
    ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), cat),
])

def make_model(name, estimator, grid):
    return Model(name=name, estimator_name=name, estimator=estimator, model_type="classification",
                 pipeline_steps=[("pre", preprocess)], grid=grid, scoring=["roc_auc"],
                 stratify_y=True, train_size=0.6, validation_size=0.2, test_size=0.2, random_state=42, display=False)

lr = make_model("lr", LogisticRegression(max_iter=2000), [{"lr__C": [10]}])
xgb = make_model("xgb", XGBClassifier(n_estimators=300, random_state=42, n_jobs=-1, eval_metric="logloss"),
                 [{"xgb__max_depth": [2], "xgb__learning_rate": [0.03], "xgb__subsample": [0.8]}])
for m in (lr, xgb):
    m.grid_search_param_tuning(X, y)
X_train, y_train = lr.get_train_data(X, y)
X_test, y_test = lr.get_test_data(X, y)
for m in (lr, xgb):
    m.fit(X_train, y_train)

p_lr = lr.predict_proba(X_test)[:, 1]
p_xgb = xgb.predict_proba(X_test)[:, 1]
print("test rows:", len(y_test), "  cases:", int(y_test.sum()))

## 2. The confusion matrix and the four numbers

Pick a cutoff, call everyone above it positive, and count four things: true positives, false positives, false negatives, true negatives. From those, four rates, each answering a different person's question:

| Rate | Formula | Whose question |
|---|---|---|
| **Sensitivity** (recall) | TP / (TP + FN) | the clinician: of the people who have it, how many do we catch? |
| **Specificity** | TN / (TN + FP) | the lab budget: of the people who don't, how many do we correctly leave alone? |
| **PPV** (precision) | TP / (TP + FP) | the patient who was flagged: how likely is it that I actually have it? |
| **NPV** | TN / (TN + FN) | the patient who wasn't: how sure are we? |

PPV and NPV depend on the base rate; sensitivity and specificity don't. That's why a test that's 95% sensitive and 95% specific still produces mostly false alarms when the disease is rare: with a 16% base rate, even good tests flag more healthy people than sick ones at any cutoff that catches most cases.

At the default 0.5 cutoff, here is what last chapter's model does.

In [ ]:
from model_metrics import show_confusion_matrix

show_confusion_matrix(y=y_test, y_prob=p_lr, model_threshold=0.5, model_title="Logistic regression, cutoff 0.5",
                      class_labels=["no diabetes", "diabetes"], class_report=True, figsize=(5, 4))

Sensitivity 11%. The model misses nine of every ten cases. Specificity 99%: almost nobody healthy gets flagged. For a screening program this is backwards. The cutoff is the problem, not the model.

## 3. Moving the cutoff

Lower the cutoff and sensitivity rises, specificity falls, PPV falls. There is no free lunch; there is only the question of which direction the clinic wants to err. Sweep the cutoff and plot all of them.

In [ ]:
from model_metrics import plot_threshold_metrics

plot_threshold_metrics(y_test=y_test, y_prob=p_lr, model_title="Logistic regression", figsize=(9, 5),
                       lookup_metric="recall", lookup_value=0.80)

Three ways to choose a cutoff, each a different statement about costs:

1. **Youden's J** (maximize sensitivity + specificity - 1): treats the two kinds of error as equally bad. Rarely true, but a reasonable default when nobody will tell you the costs.
2. **F-beta**: a weighted harmonic mean of precision and recall; beta > 1 weights recall more. Beta = 2 says "missing a case is about twice as bad as a false alarm."
3. **A target on one metric**: "we want to catch 80% of cases; what cutoff does that, and what does it cost in false alarms?" This is how clinicians actually talk, and `model_tuner` can solve for it directly.

In [ ]:
from model_tuner import find_optimal_threshold_beta, threshold_tune
from sklearn.metrics import roc_curve, recall_score, precision_score

fpr, tpr, thr = roc_curve(y_test, p_lr)
t_youden = thr[np.argmax(tpr - fpr)]
t_f2 = threshold_tune(y_test.values, p_lr, betas=[2])
t_rec80 = find_optimal_threshold_beta(y_test.values, p_lr, target_metric="recall", target_score=0.80)

rules = {"default 0.5": 0.5, "Youden's J": t_youden, "F2 (recall weighted)": float(t_f2), "target: 80% sensitivity": float(t_rec80[0])}
rows = []
for name, t in rules.items():
    pred = (p_lr >= t).astype(int)
    rows.append({"rule": name, "cutoff": round(t, 3), "flagged": int(pred.sum()), "share flagged": pred.mean(),
                 "sensitivity": recall_score(y_test, pred), "specificity": recall_score(y_test, pred, pos_label=0),
                 "PPV": precision_score(y_test, pred)})
pd.DataFrame(rows).set_index("rule").round(3)

Read the last row: to catch about 80% of cases the cutoff drops to 0.16, right around the base rate; the model flags nearly 40% of all adults, and about one in three of those flagged actually has diabetes. Youden's J lands in nearly the same place. F2 pushes further: 90% of cases caught, but more than half of adults flagged. That is the real shape of the screening problem, and no algorithm changes it; a better model would shift the numbers a little, not change the story.

## 4. ROC and precision-recall curves

A threshold sweep makes a table. Two curves show the whole sweep at once.

**ROC**: sensitivity (y) against 1 - specificity (x), one point per cutoff. The diagonal is a coin flip. AUC is the area under it: the probability a random case outscores a random non-case. It ignores the base rate, which makes it good for comparing models and bad for telling you what a flagged patient should feel.

**Precision-recall**: PPV (y) against sensitivity (x). The horizontal baseline is the base rate (a random model's precision). This curve *does* feel the base rate, which is why it drops so steeply as you push recall. For rare outcomes it's the more honest picture. Average precision (AP) summarizes it.

In [ ]:
from model_metrics import show_roc_curve, show_pr_curve

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
show_roc_curve(y=y_test, y_prob=[p_lr, p_xgb], model_title=["Logistic regression", "XGBoost"], overlay=True, ax=axes[0], title="ROC")
show_pr_curve(y=y_test, y_prob=[p_lr, p_xgb], model_title=["Logistic regression", "XGBoost"], overlay=True, ax=axes[1], title="Precision-recall")
plt.tight_layout(); plt.show()

The two models' curves sit nearly on top of each other. Which raises the question: is XGBoost's slightly higher AUC real, or noise from which 1,113 people landed in the test set?

## 5. Comparing two models honestly

The **DeLong test** gives a p-value for the difference between two AUCs computed on the same test set, accounting for the fact that they're correlated (same people). It's the right tool when someone says "our new model is better" and shows you two AUCs to the third decimal.

In [ ]:
from model_metrics import delong_roc_test

delong_roc_test(y_test, p_lr, p_xgb, model_names=["Logistic regression", "XGBoost"])

No evidence the AUCs differ. The simpler model stays.

## 6. Calibration: does 23% mean 23%?

AUC judges *ranking*: cases above non-cases. It says nothing about whether the numbers are right. A model could rank perfectly while saying everyone has a 2% or a 3% chance. For a cutoff stated as a probability ("order the test above 15%") the probabilities have to mean what they say.

**Calibration curve**: bin patients by predicted probability, and in each bin compare the predicted average to the observed share of cases. Points on the diagonal are calibrated. The **Brier score** is the mean squared error of the probabilities; lower is better, and 0.13 (predicting the base rate for everyone) is the number to beat.

In [ ]:
from model_metrics import show_calibration_curve

show_calibration_curve(y=y_test, y_prob=[p_lr, p_xgb], model_title=["Logistic regression", "XGBoost"], overlay=True, bins=10, figsize=(6, 6))

Both track the diagonal through the range where most people sit (under 40%), and both get noisy above that because few test-set patients have predictions that high. Logistic regression tends to be well calibrated by construction; boosted trees often aren't and need post-hoc calibration (`calibrate=True` in `model_tuner` fits a Platt or isotonic correction on the validation set). Here the boosted model is fine, because it's shallow.

## 7. Error bars on everything

Every number above came from one test set of 1,113 people. Chapter 3's lesson applies: a statistic without an interval is a guess. Bootstrap the test set and recompute.

In [ ]:
from model_tuner import evaluate_bootstrap_metrics

boot = evaluate_bootstrap_metrics(
    y=y_test.values, y_pred_prob=p_lr, n_samples=len(y_test), num_resamples=1000,
    metrics=["roc_auc", "average_precision", "recall", "precision", "specificity"],
    threshold=float(t_rec80[0]), random_state=42, ci_method="percentile",
)
boot

AUC 0.80 (0.76 to 0.83); at the 80%-sensitivity cutoff, sensitivity 0.78 (0.72 to 0.85) and PPV 0.32 (0.28 to 0.37). Those percentile intervals go in the write-up next to the point estimates. (The function's default `ci_method="t"` reports the standard error of the bootstrap *mean*, which is a different and much narrower thing; use `"percentile"` for the interval you mean.) A reviewer who sees "AUC 0.796" and one who sees "AUC 0.80 (0.76 to 0.83)" draw different conclusions about whether it beats a published 0.78.

## 8. The screening rule

Everything above, translated into what the medical director asked for. For a few candidate cutoffs: what share of adults get flagged, what share of cases get caught, and how many tests it takes to find one case (**number needed to screen**).

In [ ]:
rows = []
for t in [0.10, 0.14, 0.20, 0.30, 0.50]:
    pred = (p_lr >= t).astype(int)
    tp = int(((pred == 1) & (y_test == 1)).sum())
    rows.append({"cutoff": t, "share of adults flagged": pred.mean(), "share of cases caught": recall_score(y_test, pred),
                 "tests per case found": pred.sum() / max(tp, 1), "cases missed per 1,000 adults": 1000 * ((pred == 0) & (y_test == 1)).mean()})
pd.DataFrame(rows).set_index("cutoff").round(2)

In [ ]:
from model_metrics import show_gain_chart

# The same information as a picture: if the clinic can only test the top X% by risk, what share of cases does that reach?
show_gain_chart(y=y_test, y_prob=p_lr, model_title="Logistic regression", figsize=(6, 5), title="Cumulative gain: cases reached by testing the highest-risk share of adults")

The recommendation, in the language of the decision: *screen adults whose predicted probability is 14% or higher. That flags about 43% of adults, catches four in five cases, and costs between three and four tests per case found; it misses about 30 cases per 1,000 adults. If testing capacity is limited, testing the top 30% by predicted risk (cutoff 0.20) still reaches 70% of cases.* The director picks the row; the data scientist's job was to make the rows honest.

Compare that to where we started in chapter 1, with a plot of prevalence by age and BMI. The model does the same thing with eight variables instead of two, with intervals, and with a dial the clinic can turn.

## What to carry forward

- Accuracy answers nobody's question when classes are imbalanced. Report sensitivity, specificity, PPV at the cutoff you'd use.
- The cutoff is a policy decision; solve for it from a stated target or cost ratio, don't inherit 0.5.
- ROC for comparing models; PR for understanding what flagged patients experience; DeLong before claiming one AUC beats another.
- Check calibration whenever a probability is going to be read as a probability.
- Bootstrap the test set; report intervals.
- End with the rule, in the decision-maker's units.

## Looking ahead
- **Reading**: ISLR chapter 12 (unsupervised learning), 12.1 through 12.4. Next chapter there is no outcome variable at all.